In [1]:
import sys 
sys.path.insert(0, '../')

import numpy as np
import matplotlib.pyplot as plt

In [2]:
from src.utils import load_signals_and_arousals

example_patient_path = "../data/raw/tr03-0005"

signals, arousals, sleep_stages = load_signals_and_arousals(example_patient_path, include_sleep_stages=True)

..\data\raw\tr03-0005\tr03-0005.mat ..\data\raw\tr03-0005\tr03-0005-arousal.mat


In [3]:
def find_stable_blocks(stage_mask: np.ndarray, min_len_samp: int) -> list:
    """
    Finds contiguous blocks of 1s in a boolean array that are >= min_len_samp.
    Returns a list of (onset_index, offset_index) tuples.
    """
    # Pad with 0s to easily detect edges if the mask starts or ends with 1
    edges = np.diff(np.concatenate(([0], stage_mask, [0])))
    
    # 1 indicates a transition from 0 to 1 (onset)
    # -1 indicates a transition from 1 to 0 (offset)
    onsets = np.where(edges == 1)[0]
    offsets = np.where(edges == -1)[0]
    
    valid_blocks = []
    for on, off in zip(onsets, offsets):
        if (off - on) >= min_len_samp:
            valid_blocks.append((on, off))
            
    return valid_blocks

In [4]:
blocks = find_stable_blocks(sleep_stages['wake'], min_len_samp=200*30)
blocks

[(np.int64(71999), np.int64(119999)),
 (np.int64(131999), np.int64(137999)),
 (np.int64(167999), np.int64(173999)),
 (np.int64(215999), np.int64(227999)),
 (np.int64(233999), np.int64(245999)),
 (np.int64(251999), np.int64(281999)),
 (np.int64(287999), np.int64(329999)),
 (np.int64(827999), np.int64(839999)),
 (np.int64(983999), np.int64(989999)),
 (np.int64(1361999), np.int64(1373999)),
 (np.int64(1631999), np.int64(1637999)),
 (np.int64(1661999), np.int64(1667999)),
 (np.int64(2039999), np.int64(2057999)),
 (np.int64(2537999), np.int64(2561999)),
 (np.int64(2567999), np.int64(2573999)),
 (np.int64(2813999), np.int64(2819999)),
 (np.int64(3083999), np.int64(3155999)),
 (np.int64(3167999), np.int64(3173999)),
 (np.int64(3191999), np.int64(3215999)),
 (np.int64(3221999), np.int64(3269999)),
 (np.int64(3275999), np.int64(3287999)),
 (np.int64(3293999), np.int64(3353999)),
 (np.int64(3359999), np.int64(4355999)),
 (np.int64(4367999), np.int64(4391999)),
 (np.int64(4397999), np.int64(44699

In [5]:
STAGE_TO_INT = {
    'wake': 0,
    'nonrem1': 1,
    'nonrem2': 2,
    'nonrem3': 3,
    'rem': 4
}

def extract_sleep_stage_windows(
    sleep_stages: dict, 
    full_spectral_timeline: np.ndarray, 
    fs: int, 
    hop_length: int,
    win_sec: int = 30
) -> tuple[np.ndarray, np.ndarray]:
    """
    Extracts matching STFT features from stable sleep stages.
    Returns:
        spectral_features: np.ndarray of shape (n_windows, time_steps, features)
        labels: np.ndarray of shape (n_windows, n_classes) as one-hot vectors
    """
    # 30 seconds converted to raw samples
    win_samp = win_sec * fs
    
    # Standard 30s epochs do not overlap, so step equals the window size
    step_samp = win_samp 
    
    all_features = []
    all_labels = []
    
    for stage_name, mask in sleep_stages.items():
        # Ignore undefined or unmapped stages
        if stage_name == 'undefined' or stage_name not in STAGE_TO_INT:
            continue
            
        stage_idx = STAGE_TO_INT[stage_name]
        
        # 1. Get contiguous blocks for this stage
        blocks = find_stable_blocks(mask, win_samp)
        
        # 2. Extract consecutive 30-second windows
        for on, off in blocks:
            for start_idx in range(on, off - win_samp + 1, step_samp):
                end_idx = start_idx + win_samp
                
                # Convert raw sample indices to STFT step indices
                start_step_idx = start_idx // hop_length
                end_step_idx = end_idx // hop_length
                
                # Slice STFT Features
                spec_win = full_spectral_timeline[start_step_idx:end_step_idx, :]
                
                all_features.append(spec_win)
                all_labels.append(stage_idx)
                
    # 3. Stack everything into flat numpy arrays
    spectral_features = np.array(all_features)
    labels_int = np.array(all_labels)
    
    # 4. One-hot encode the labels using an identity matrix trick
    num_classes = len(STAGE_TO_INT)
    labels_onehot = np.eye(num_classes)[labels_int]
    
    return spectral_features, labels_onehot

In [7]:
from src.dsp import compute_full_recording_bandpower

full_spectral_timeline = compute_full_recording_bandpower(signals, fs=200, hop_length=200)
spectral_windows, labels = extract_sleep_stage_windows(sleep_stages, full_spectral_timeline, fs=200, hop_length=200)

In [13]:
spectral_windows[0]

array([[1.19341408e+02, 9.44023895e+01, 1.18465744e+02, ...,
        1.63622772e+02, 6.13164231e-02, 5.02454132e+02],
       [1.17941360e+02, 9.46429901e+01, 1.16433731e+02, ...,
        1.65936890e+02, 6.13164231e-02, 4.87279663e+02],
       [1.35688034e+02, 1.01025467e+02, 1.35877426e+02, ...,
        2.35133026e+02, 6.13164231e-02, 4.93354553e+02],
       ...,
       [1.08308228e+02, 1.03244659e+02, 1.07153824e+02, ...,
        1.87137039e+02, 4.82778351e+02, 4.87740906e+02],
       [9.47912598e+01, 1.05227982e+02, 9.60024414e+01, ...,
        1.24211426e+02, 6.13164231e-02, 4.85822113e+02],
       [9.26336670e+01, 9.88644104e+01, 9.63075104e+01, ...,
        1.26133369e+02, 6.13164231e-02, 4.76291687e+02]],
      shape=(30, 65), dtype=float32)

In [ ]:
train_path = "../data/processed/train.npz"
test_path = "../data/processed/test.npz"

def load_data(npz_path):
    data = np.load(npz_path)
    eeg_windows = data['eeg_windows']
    context_windows = data['context_windows']
